# Experiments: ViT Variants, Augmentation, Input Size, Loss Weights

**Dataset:** `Kidney Images Dataset/` in Google Drive with `train/`, `valid/`, `test/` subfolders,
each containing `images/` (jpg) and `labels/` (YOLO `.txt` format).

**Metrics:** Precision, Recall, F1-score, mAP@0.5, mAP@0.5:0.95, Accuracy

**Configurations evaluated:**
1. ViT-Small/16 backbone
2. ViT-Base/16 backbone (proposed model)
3. ViT-Large/16 backbone
4. ViT-Base/16 without data augmentation
5. ViT-Base/16 with input size 160x160
6. ViT-Base/16 with default loss weights (no class balancing)

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Install Dependencies

In [ ]:
!pip install transformers scikit-learn seaborn matplotlib --quiet
print('Done.')

## Step 3: Run All Experiments



In [ ]:
import os, gc, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import ViTModel
from huggingface_hub import notebook_login # Added this line
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    accuracy_score, confusion_matrix, average_precision_score
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# Added this line to prompt for Hugging Face login
notebook_login()

DATASET_ROOT = '/content/drive/MyDrive/Kidney Images Dataset'

TRAIN_IMG  = os.path.join(DATASET_ROOT, 'train', 'images')
TRAIN_LBL  = os.path.join(DATASET_ROOT, 'train', 'labels')
VALID_IMG  = os.path.join(DATASET_ROOT, 'valid', 'images')
VALID_LBL  = os.path.join(DATASET_ROOT, 'valid', 'labels')
TEST_IMG   = os.path.join(DATASET_ROOT, 'test',  'images')
TEST_LBL   = os.path.join(DATASET_ROOT, 'test',  'labels')
OUTPUT_DIR = os.path.join(DATASET_ROOT, 'ablation_results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

EPOCHS     = 50
PHASE1_EPS = 15
PHASE2_EPS = EPOCHS - PHASE1_EPS

print('Checking directories...')
for lbl, path in [('train/images', TRAIN_IMG), ('train/labels', TRAIN_LBL),
                   ('valid/images', VALID_IMG), ('valid/labels', VALID_LBL),
                   ('test/images',  TEST_IMG),  ('test/labels',  TEST_LBL)]:
    if not os.path.isdir(path):
        raise FileNotFoundError(f'MISSING: {lbl} -> {path}')
    print(f'  OK  {lbl}')


def parse_yolo_txt(lbl_path):
    boxes = []
    if lbl_path is None or not os.path.isfile(str(lbl_path)):
        return boxes
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                try:
                    boxes.append(tuple(float(v) for v in parts[:5]))
                except ValueError:
                    pass
    return boxes


def build_pairs(img_dir, lbl_dir):
    lbl_lookup = {p.stem: p for p in Path(lbl_dir).glob('*.txt')}
    pairs = []
    for ext in ('*.jpg', '*.jpeg', '*.png'):
        for img_path in sorted(Path(img_dir).glob(ext)):
            boxes = parse_yolo_txt(lbl_lookup.get(img_path.stem))
            label = 1 if len(boxes) > 0 else 0
            pairs.append((str(img_path), label, boxes))
    return pairs


train_pairs = build_pairs(TRAIN_IMG, TRAIN_LBL)
valid_pairs = build_pairs(VALID_IMG, VALID_LBL)
test_pairs  = build_pairs(TEST_IMG,  TEST_LBL)

print('Dataset label distribution:')
print(f'  {"Split":<8} {"Total":>6} {"Stone":>8} {"Normal":>8}')
for sname, pairs in [('Train', train_pairs), ('Valid', valid_pairs), ('Test', test_pairs)]:
    s = sum(l for _, l, _ in pairs)
    n = len(pairs) - s
    print(f'  {sname:<8} {len(pairs):>6} {s:>8} {n:>8}')
    if len(pairs) == 0:
        raise ValueError(f'{sname} has 0 samples. Check DATASET_ROOT.')

n_stone  = sum(l for _, l, _ in train_pairs)
n_normal = len(train_pairs) - n_stone
BALANCED_WEIGHT = float(n_normal) / max(float(n_stone), 1.0)
print(f'pos_weight: {BALANCED_WEIGHT:.3f}  (stone={n_stone}, normal={n_normal})')


class KidneyStoneDataset(Dataset):
    def __init__(self, pairs, img_size=224, augment=False):
        self.pairs = pairs
        base = [
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
        aug = [
            T.Resize((img_size, img_size)),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.RandomRotation(15),
            T.ColorJitter(brightness=0.3, contrast=0.3),
            T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
        self.transform = T.Compose(aug if augment else base)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, label, _ = self.pairs[idx]
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), color=0)
        return self.transform(img), torch.tensor(label, dtype=torch.float32)


class ViTBackedClassifier(nn.Module):
    def __init__(self, vit_model_id, out_channels=256, dropout_p=0.1):
        super().__init__()
        self.vit = ViTModel.from_pretrained(vit_model_id, add_pooling_layer=False)
        self.hidden_size  = self.vit.config.hidden_size
        self.vit_img_size = self.vit.config.image_size
        self.pos_dropout  = nn.Dropout(p=dropout_p)
        self.proj = nn.Conv2d(self.hidden_size, out_channels, kernel_size=1)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(out_channels, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 1),
        )
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        H_in   = x.shape[2]
        interp = H_in != self.vit_img_size
        out    = self.vit(pixel_values=x, interpolate_pos_encoding=interp)
        tokens = out.last_hidden_state[:, 1:, :]
        tokens = self.pos_dropout(tokens)
        B, N, D = tokens.shape
        H = W = int(N ** 0.5)
        F = tokens.permute(0, 2, 1).reshape(B, D, H, W)
        F = self.proj(F)
        F = self.pool(F)
        return self.head(F).squeeze(1)


def run_epoch(model, loader, criterion, optimizer=None, threshold=0.5):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, all_labels, all_preds, all_probs = 0.0, [], [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs   = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            all_probs.extend(probs.tolist())
            all_labels.extend(labels.cpu().numpy().astype(int).tolist())
            all_preds.extend((probs >= threshold).astype(int).tolist())
    avg_loss = total_loss / max(len(loader.dataset), 1)
    p  = float(precision_score(all_labels, all_preds, zero_division=0))
    r  = float(recall_score(all_labels,    all_preds, zero_division=0))
    f1 = float(f1_score(all_labels,        all_preds, zero_division=0))
    a  = float(accuracy_score(all_labels,  all_preds))
    return avg_loss, p, r, f1, a, all_labels, all_probs


def compute_map(all_labels, all_probs):
    if len(np.unique(all_labels)) < 2:
        return 0.0, 0.0
    ap50 = float(average_precision_score(all_labels, all_probs))
    aps  = []
    for t in np.arange(0.50, 1.00, 0.05):
        preds = (np.array(all_probs) >= t).astype(int)
        p_ = float(precision_score(all_labels, preds, zero_division=0))
        r_ = float(recall_score(all_labels,    preds, zero_division=0))
        aps.append(p_ * r_)
    return ap50, float(np.mean(aps))


def train_and_evaluate(cfg, tr_ld, vl_ld, te_ld):
    model     = ViTBackedClassifier(cfg['vit_model_id']).to(DEVICE)
    pw        = torch.tensor([cfg['pos_weight']], dtype=torch.float32).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    hist_keys = ['train_loss', 'train_prec', 'train_rec', 'train_f1', 'train_acc',
                 'val_loss',   'val_prec',   'val_rec',   'val_f1',   'val_acc']
    history   = {k: [] for k in hist_keys}
    best_f1   = -1.0
    best_w    = None

    for p in model.vit.parameters():
        p.requires_grad = False
    trainable = list(filter(lambda p: p.requires_grad, model.parameters()))
    opt1   = optim.AdamW(trainable, lr=1e-3, weight_decay=1e-4)
    sched1 = optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=PHASE1_EPS, eta_min=1e-5)
    print(f'  Phase 1: {PHASE1_EPS} epochs, ViT encoder frozen')
    for epoch in range(1, PHASE1_EPS + 1):
        tl, tp, tr, tf, ta, _, _ = run_epoch(model, tr_ld, criterion, opt1)
        vl, vp, vr, vf, va, _, _ = run_epoch(model, vl_ld, criterion)
        sched1.step()
        for k, v in zip(hist_keys, [tl, tp, tr, tf, ta, vl, vp, vr, vf, va]):
            history[k].append(v)
        if vf > best_f1:
            best_f1 = vf
            best_w  = {k: v.clone() for k, v in model.state_dict().items()}
        if epoch % 5 == 0 or epoch == 1:
            print(f'    [{epoch:2d}/{PHASE1_EPS}] ValLoss={vl:.4f} ValF1={vf:.4f} ValAcc={va:.4f}')

    for p in model.parameters():
        p.requires_grad = True
    opt2 = optim.AdamW([
        {'params': model.vit.parameters(),  'lr': 1e-5},
        {'params': model.proj.parameters(), 'lr': 1e-4},
        {'params': model.head.parameters(), 'lr': 1e-4},
    ], weight_decay=1e-4)
    sched2 = optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=PHASE2_EPS, eta_min=1e-6)
    print(f'  Phase 2: {PHASE2_EPS} epochs, all layers unfrozen')
    for epoch in range(1, PHASE2_EPS + 1):
        tl, tp, tr, tf, ta, _, _ = run_epoch(model, tr_ld, criterion, opt2)
        vl, vp, vr, vf, va, _, _ = run_epoch(model, vl_ld, criterion)
        sched2.step()
        for k, v in zip(hist_keys, [tl, tp, tr, tf, ta, vl, vp, vr, vf, va]):
            history[k].append(v)
        if vf > best_f1:
            best_f1 = vf
            best_w  = {k: v.clone() for k, v in model.state_dict().items()}
        if epoch % 5 == 0 or epoch == 1:
            print(f'    [{epoch:2d}/{PHASE2_EPS}] ValLoss={vl:.4f} ValF1={vf:.4f} ValAcc={va:.4f}')

    if best_w is not None:
        model.load_state_dict(best_w)
    print(f'  Best Val F1: {best_f1:.4f}')

    _, p, r, f1, acc, true_lbl, probs = run_epoch(model, te_ld, criterion)
    map50, map5095 = compute_map(true_lbl, probs)
    preds = (np.array(probs) >= 0.5).astype(int).tolist()
    metrics = {
        'Precision':    round(p,       4),
        'Recall':       round(r,       4),
        'F1-score':     round(f1,      4),
        'mAP@0.5':      round(map50,   4),
        'mAP@0.5:0.95': round(map5095, 4),
        'Accuracy':     round(acc,     4),
        'true_labels':  true_lbl,
        'predictions':  preds,
    }
    return model, metrics, history


ABLATION_CONFIGS = [
    {
        'name':         'ViT-Small/16 (224x224, augmented, optimized weights)',
        'short':        'ViT-Small/16',
        'vit_model_id': 'google/vit-small-patch16-224',
        'img_size':     224,
        'augment':      True,
        'pos_weight':   BALANCED_WEIGHT,
        'batch_size':   16,
    },
    {
        'name':         'ViT-Base/16 Proposed (224x224, augmented, optimized weights)',
        'short':        'ViT-Base/16 (Proposed)',
        'vit_model_id': 'google/vit-base-patch16-224-in21k',
        'img_size':     224,
        'augment':      True,
        'pos_weight':   BALANCED_WEIGHT,
        'batch_size':   16,
    },
    {
        'name':         'ViT-Large/16 (224x224, augmented, optimized weights)',
        'short':        'ViT-Large/16',
        'vit_model_id': 'google/vit-large-patch16-224-in21k',
        'img_size':     224,
        'augment':      True,
        'pos_weight':   BALANCED_WEIGHT,
        'batch_size':   8,
    },
    {
        'name':         'ViT-Base/16 No Augmentation (224x224, optimized weights)',
        'short':        'No Augmentation',
        'vit_model_id': 'google/vit-base-patch16-224-in21k',
        'img_size':     224,
        'augment':      False,
        'pos_weight':   BALANCED_WEIGHT,
        'batch_size':   16,
    },
    {
        'name':         'ViT-Base/16 Input 160x160 (augmented, optimized weights)',
        'short':        'Input 160x160',
        'vit_model_id': 'google/vit-base-patch16-224-in21k',
        'img_size':     160,
        'augment':      True,
        'pos_weight':   BALANCED_WEIGHT,
        'batch_size':   16,
    },
    {
        'name':         'ViT-Base/16 Default Loss Weights (224x224, augmented)',
        'short':        'Default Loss Weights',
        'vit_model_id': 'google/vit-base-patch16-224-in21k',
        'img_size':     224,
        'augment':      True,
        'pos_weight':   1.0,
        'batch_size':   16,
    },
]

COLORS        = ['#4472C4', '#ED7D31', '#A9D18E', '#FFC000', '#5B9BD5', '#FF6B6B']
all_results   = []
all_histories = []

for idx, cfg in enumerate(ABLATION_CONFIGS):
    print(f'\n{"="*68}')
    print(f'Config [{idx+1}/{len(ABLATION_CONFIGS)}]: {cfg["name"]}')
    print('='*68)
    tr_ds = KidneyStoneDataset(train_pairs, cfg['img_size'], augment=cfg['augment'])
    vl_ds = KidneyStoneDataset(valid_pairs, cfg['img_size'], augment=False)
    te_ds = KidneyStoneDataset(test_pairs,  cfg['img_size'], augment=False)
    bs    = cfg['batch_size']
    tr_ld = DataLoader(tr_ds, bs, shuffle=True,  num_workers=2, pin_memory=True)
    vl_ld = DataLoader(vl_ds, bs, shuffle=False, num_workers=2, pin_memory=True)
    te_ld = DataLoader(te_ds, bs, shuffle=False, num_workers=2, pin_memory=True)
    model, metrics, history = train_and_evaluate(cfg, tr_ld, vl_ld, te_ld)
    print('  Test Results:')
    for mn in ['Precision', 'Recall', 'F1-score', 'mAP@0.5', 'mAP@0.5:0.95', 'Accuracy']:
        print(f'    {mn:<16}: {metrics[mn]:.4f}')
    all_results.append({
        'Configuration': cfg['name'],
        'Short':         cfg['short'],
        'Precision':     metrics['Precision'],
        'Recall':        metrics['Recall'],
        'F1-score':      metrics['F1-score'],
        'mAP@0.5':       metrics['mAP@0.5'],
        'mAP@0.5:0.95':  metrics['mAP@0.5:0.95'],
        'Accuracy':      metrics['Accuracy'],
        'true_labels':   metrics['true_labels'],
        'predictions':   metrics['predictions'],
    })
    all_histories.append({'short': cfg['short'], 'history': history})
    safe_name = cfg['short'].replace('/', '_').replace(' ', '_')
    wt_path   = os.path.join(OUTPUT_DIR, f'weights_{idx+1}_{safe_name}.pt')
    torch.save(model.state_dict(), wt_path)
    print(f'  Weights saved: {os.path.basename(wt_path)}')
    del model, tr_ds, vl_ds, te_ds, tr_ld, vl_ld, te_ld
    gc.collect()
    torch.cuda.empty_cache()

METRIC_COLS = ['Precision', 'Recall', 'F1-score', 'mAP@0.5', 'mAP@0.5:0.95', 'Accuracy']

results_df = pd.DataFrame([{k: r[k] for k in ['Configuration'] + METRIC_COLS}
                            for r in all_results])
csv_path = os.path.join(OUTPUT_DIR, 'ablation_all_results.csv')
results_df.to_csv(csv_path, index=False)
print('\n' + '='*80)
print('ABLATION STUDY COMPLETE RESULTS')
print('='*80)
hdr = f'  {"Short Label":<28}' + '  '.join(f'{m:>12}' for m in METRIC_COLS)
print(hdr)
print('  ' + '-'*80)
for r in all_results:
    mk   = '*' if 'Proposed' in r['Short'] else ' '
    vals = '  '.join(f'{r[m]:>12.4f}' for m in METRIC_COLS)
    print(f'{mk} {r["Short"]:<28} {vals}')
print('(* = proposed model from manuscript)')
print(f'Saved: {csv_path}')

n_cfgs = len(ABLATION_CONFIGS)
x, w   = np.arange(len(METRIC_COLS)), 0.13
fig, ax = plt.subplots(figsize=(17, 7))
for i, (result, color) in enumerate(zip(all_results, COLORS)):
    vals   = [result[m] for m in METRIC_COLS]
    offset = (i - n_cfgs / 2.0 + 0.5) * w
    bars   = ax.bar(x + offset, vals, w, label=result['Short'],
                    color=color, edgecolor='white', linewidth=0.4)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.003,
                f'{h:.3f}', ha='center', va='bottom', fontsize=5, rotation=90)
ax.set_xticks(x)
ax.set_xticklabels(METRIC_COLS, fontsize=11)
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim(0, 1.25)
ax.set_title('Ablation Study: All Configurations on Test Set', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=8, ncol=2, framealpha=0.9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
bar_path = os.path.join(OUTPUT_DIR, 'ablation_bar_chart.pdf')
plt.savefig(bar_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {bar_path}')

n_cols  = 3
n_rows  = int(np.ceil(n_cfgs / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 5))
axes_flat = axes.flatten()
for i, result in enumerate(all_results):
    ax   = axes_flat[i]
    cm   = confusion_matrix(result['true_labels'], result['predictions'])
    cm_n = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100
    sns.heatmap(cm_n, annot=True, fmt='.1f', cmap='Blues', ax=ax,
                xticklabels=['No Stone', 'Stone'],
                yticklabels=['No Stone', 'Stone'],
                cbar_kws={'label': 'Percent within actual class'})
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('Actual',    fontsize=10)
    mk = ' (*)' if 'Proposed' in result['Short'] else ''
    ax.set_title(result['Short'] + mk, fontsize=10, fontweight='bold')
for ax in axes_flat[n_cfgs:]:
    ax.set_visible(False)
plt.suptitle('Confusion Matrices: All Ablation Configurations (* = proposed)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, 'ablation_confusion_matrices.pdf')
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {cm_path}')

ep   = range(1, EPOCHS + 1)
fig, axes = plt.subplots(2, n_cfgs, figsize=(n_cfgs * 4, 9))
for i, (hdata, color) in enumerate(zip(all_histories, COLORS)):
    hist = hdata['history']
    ax0  = axes[0, i]
    ax0.plot(ep, hist['train_f1'], color=color, linewidth=1.8, label='Train')
    ax0.plot(ep, hist['val_f1'],   color=color, linewidth=1.8,
             linestyle='--', alpha=0.7, label='Val')
    ax0.axvline(PHASE1_EPS, color='gray', linestyle=':', linewidth=1.2,
                label=f'Ph2 start')
    ax0.set_ylim(0, 1.05)
    ax0.set_ylabel('F1-score', fontsize=9)
    ax0.set_title(hdata['short'], fontsize=9, fontweight='bold')
    ax0.legend(fontsize=7)
    ax0.grid(True, alpha=0.3)
    ax1 = axes[1, i]
    ax1.plot(ep, hist['train_loss'], color=color, linewidth=1.8, label='Train')
    ax1.plot(ep, hist['val_loss'],   color=color, linewidth=1.8,
             linestyle='--', alpha=0.7, label='Val')
    ax1.axvline(PHASE1_EPS, color='gray', linestyle=':', linewidth=1.2)
    ax1.set_ylabel('Loss',  fontsize=9)
    ax1.set_xlabel('Epoch', fontsize=9)
    ax1.legend(fontsize=7)
    ax1.grid(True, alpha=0.3)
plt.suptitle('Training and Validation Curves: All Ablation Configurations',
             fontsize=11, fontweight='bold')
plt.tight_layout()
tc_path = os.path.join(OUTPUT_DIR, 'ablation_training_curves.pdf')
plt.savefig(tc_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {tc_path}')

print('\n' + '='*80)
print('FINAL SUMMARY')
print('='*80)
hdr = (f'  {"Config":<30} '
       f'{"Prec":>6} {"Rec":>6} {"F1":>6} '
       f'{"mAP50":>7} {"mAP5095":>8} {"Acc":>7}')
print(hdr)
print('  ' + '-'*76)
for r in all_results:
    mk = '*' if 'Proposed' in r['Short'] else ' '
    print(f'{mk} {r["Short"]:<30} '
          f'{r["Precision"]:>6.4f} '
          f'{r["Recall"]:>6.4f} '
          f'{r["F1-score"]:>6.4f} '
          f'{r["mAP@0.5"]:>7.4f} '
          f'{r["mAP@0.5:0.95"]:>8.4f} '
          f'{r["Accuracy"]:>7.4f}')
print('='*80)
print('(* = proposed model from manuscript)')
print(f'All outputs saved to: {OUTPUT_DIR}')